In [0]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Making dataframes for all the cleaned csv files

ratings_df = pd.read_csv('ratings.csv')
reviews_df = pd.read_csv('reviews.csv')
subscribers_df = pd.read_csv('subscribers.csv')
titles_df = pd.read_csv('titles.csv')
watch_history_df = pd.read_csv('watch_history.csv')
watchlist_df = pd.read_csv('watchlist.csv')

In [0]:
ratings_df.info()

In [0]:
# From ratings.csv convert the rating_date type from object to datetime
ratings_df['rating_date'] = pd.to_datetime(ratings_df['rating_date'])
ratings_df.info()

In [0]:
reviews_df.info()

In [0]:
# from reviews_df convert the review_date from object to datetime
reviews_df['review_date'] = pd.to_datetime(reviews_df['review_date'])
reviews_df.info()

In [0]:
subscribers_df.info()

In [0]:
# from subscribers.csv convert the signup_date and churn_date from object to datetime
subscribers_df['signup_date'] = pd.to_datetime(subscribers_df['signup_date'])
subscribers_df['churn_date'] = pd.to_datetime(subscribers_df['churn_date'])

subscribers_df.info()

In [0]:
titles_df.info()

In [0]:
# in titles_df the date_added and license_expiry field should be of datetime type
titles_df['date_added'] = pd.to_datetime(titles_df['date_added'])
titles_df['license_expiry'] = pd.to_datetime(titles_df['license_expiry'])

titles_df.info()

In [0]:
watch_history_df.info()

In [0]:
# in watch_history_df, watch_date should of datetime type
watch_history_df['watch_date'] = pd.to_datetime(watch_history_df['watch_date'])

watch_history_df.info()

In [0]:
watchlist_df.info()

In [0]:
# in watchlist_df, added_date should be of datetime type
watchlist_df['added_date'] = pd.to_datetime(watchlist_df['added_date'])

watchlist_df.info()

# Calculating total watch hours

In [0]:

# How to calculate: Sum(watch_druation_min) / 60
# Total engagement / Size of Platform

# 2. Calculating Total Watch Hours: Sum(watch_duration_min) / 60
total_watch_minutes = watch_history_df['watch_duration_min'].sum()
total_watch_hours = total_watch_minutes / 60.0

# 3. Defining Platform Size metrics
unique_subscribers = watch_history_df['subscriber_id'].nunique()  # User Base Size
catalog_size = watch_history_df['title_id'].nunique()             # Catalog / Title Base Size
total_sessions = len(watch_history_df)                            # Total Stream / Session Count

# 4. Total Engagement / Size of Platform
watch_hours_per_subscriber = total_watch_hours / unique_subscribers
watch_hours_per_title = total_watch_hours / catalog_size
watch_hours_per_session = total_watch_hours / total_sessions

# Display the results
print("================ PLATFORM ENGAGEMENT METRICS ================")
print(f"Total Watch Minutes        : {total_watch_minutes:,.2f} mins")
print(f"Total Watch Hours          : {total_watch_hours:,.2f} hours\n")

print(f"Platform Size (Subscribers): {unique_subscribers:,} active subscribers")
print(f"Engagement per Subscriber  : {watch_hours_per_subscriber:,.2f} hours / user\n")

print(f"Platform Size (Catalog)    : {catalog_size:,} unique titles")
print(f"Engagement per Title       : {watch_hours_per_title:,.2f} hours / title\n")

print(f"Total Sessions (Plays)     : {total_sessions:,} streams")
print(f"Engagement per Session     : {watch_hours_per_session:,.2f} hours / stream")
print("=============================================================")


# Active Rate

In [0]:

# How to calculate (Active Subscribers + Total) * 100
# Target > 70%. Health of the subscribers rate

# 1. Total and Active subscriber counts
total_subscribers = len(subscribers_df)
active_subscribers = subscribers_df['is_active'].sum()  # counts True values

# 2. Calculate Active Rate (%)
active_rate = (active_subscribers / total_subscribers) * 100

# 3. Evaluate against the 70% threshold
target_met = active_rate > 70.0
health_status = "Healthy (Above Target)" if target_met else "At Risk (Below Target)"

# Output results
print("================ SUBSCRIBER HEALTH METRICS ================")
print(f"Total Subscribers   : {total_subscribers:,}")
print(f"Active Subscribers  : {active_subscribers:,}")
print(f"Inactive / Churned  : {total_subscribers - active_subscribers:,}")
print(f"Active Rate         : {active_rate:.2f}%")
print(f"Target Benchmark    : > 70.0%")
print(f"Health Status       : {health_status}")
print("===========================================================")

# Churn Rate

In [0]:

# How to calculate: (Inactive / Total) * 100
# Target < 30%. Churn Rate

# 1. Total and Inactive subscriber counts
total_subscribers = len(subscribers_df)
inactive_subscribers = total_subscribers - subscribers_df['is_active'].sum()  # counts False values

# 2. Calculate Churn Rate (%)
churn_rate = (inactive_subscribers / total_subscribers) * 100

# 3. Evaluate against the 30% threshold
target_met = churn_rate < 30.0
churn_status = "Healthy (Below Target)" if target_met else "At Risk (Above Target)"

# Output results
print("================ SUBSCRIBER CHURN METRICS ================")
print(f"Total Subscribers   : {total_subscribers:,}")
print(f"Active Subscribers  : {total_subscribers - inactive_subscribers:,}")
print(f"Inactive Subscribers: {inactive_subscribers:,}")
print(f"Churn Rate          : {churn_rate:.2f}%")
print(f"Churn Status        : {churn_status}")
print("===========================================================")

# Avg Completion Rate

In [0]:

# Avg(completion_pct)
# Target > 60%. Are people finishing content?

# 2. Calculate Average Completion Rate
avg_completion_pct = watch_history_df['completion_pct'].mean()
median_completion_pct = watch_history_df['completion_pct'].median()

# Optional: Full completion rate (completion_pct >= 90%)
full_completion_rate = (watch_history_df['completed'].sum() / len(watch_history_df)) * 100

# 3. Evaluate against benchmark (> 60%)
target_met = avg_completion_pct > 60.0
status = "Healthy / Above Target" if target_met else "At Risk / Below Target"

# Output results
print("================ CONTENT COMPLETION METRICS ================")
print(f"Total Stream Sessions     : {len(watch_history_df):,}")
print(f"Average Completion Rate   : {avg_completion_pct:.2f}%")
print(f"Median Completion Rate    : {median_completion_pct:.2f}%")
print(f"Target Benchmark          : > 60.0%")
print(f"Benchmark Status          : {status}")
print(f"Full Completion Share     : {full_completion_rate:.2f}% (streams with >= 90% watched)")
print("============================================================")

# Monthly Recurring Revenue

In [0]:

# How to calculate: SUM(monthly_price) for active subscribers
# Why it matters: Actual recurring money earned

# 1. Filter for active subscribers
active_subs = subscribers_df[subscribers_df['is_active'] == True]

# 2. Calculate Monthly Recurring Revenue (MRR)
mrr = active_subs['monthly_price_usd'].sum()

# 3. Calculate Churned/Lost Monthly Revenue for comparison
churned_subs = subscribers_df[subscribers_df['is_active'] == False]
lost_mrr = churned_subs['monthly_price_usd'].sum()
total_potential_revenue = subscribers_df['monthly_price_usd'].sum()

# Output results
print("================ MONTHLY RECURRING REVENUE (MRR) ================")
print(f"Active Subscribers           : {len(active_subs):,}")
print(f"Monthly Recurring Revenue    : ${mrr:,.2f}")
print(f"Monthly Revenue Lost to Churn: ${lost_mrr:,.2f}")
print(f"Total Base Potential Revenue : ${total_potential_revenue:,.2f}")
print(f"Revenue Realization Rate     : {(mrr / total_potential_revenue) * 100:.2f}%")
print("================================================================")

# ARPU

In [0]:

# How to calculate: MRR / Active Subscribers
# Why it matters: Average revenue per active user

# 1. Filter for active subscribers
active_subs = subscribers_df[subscribers_df['is_active'] == True]

# 2. Calculate Monthly Recurring Revenue (MRR) and Active Count
mrr = active_subs['monthly_price_usd'].sum()
active_count = len(active_subs)

# 3. Calculate ARPU (MRR divided by Active Subscribers)
arpu = mrr / active_count

# Output results
print("================ ARPU METRICS ================")
print(f"Active Subscribers : {active_count:,}")
print(f"Total MRR          : ${mrr:,.2f}")
print(f"ARPU (Blended)     : ${arpu:.2f} / subscriber / month")
print("==============================================")

# Average Watch Time / Subscribers

In [0]:

# How to calculate = Total Watch Hours / Active Subscribers
# Why it matters: How engaged the active subscriber is?

# 1. Active subscribers count from subscribers_df
active_subs_count = subscribers_df[subscribers_df['is_active'] == True]['subscriber_id'].nunique()

# -------------------------------------------------------------
# METHOD 1: Active Users' Watch Time Only (Recommended)
# -------------------------------------------------------------
# Filter watch events for only active subscribers
active_sub_ids = subscribers_df[subscribers_df['is_active'] == True]['subscriber_id']
active_watches = watch_history_df[watch_history_df['subscriber_id'].isin(active_sub_ids)]

total_watch_hours_active = active_watches['watch_duration_min'].sum() / 60.0
avg_watch_time_active_only = total_watch_hours_active / active_subs_count

# -------------------------------------------------------------
# METHOD 2: Global Watch Hours / Active Subscribers
# -------------------------------------------------------------
total_watch_hours_global = watch_history_df['watch_duration_min'].sum() / 60.0
avg_watch_time_global = total_watch_hours_global / active_subs_count

# Output results
print("================ AVERAGE WATCH TIME / ACTIVE SUBSCRIBER ================")
print(f"Active Subscribers Count         : {active_subs_count:,}")
print(f"\n--- Method 1 (Active Subscribers' Watch Time Only) ---")
print(f"Active Subs Total Watch Hours    : {total_watch_hours_active:,.2f} hours")
print(f"Avg Watch Time / Active Sub      : {avg_watch_time_active_only:.2f} hours / subscriber")

print(f"\n--- Method 2 (Global Total Watch Hours / Active Subs) ---")
print(f"Global Total Watch Hours         : {total_watch_hours_global:,.2f} hours")
print(f"Avg Watch Time / Active Sub      : {avg_watch_time_global:.2f} hours / subscriber")
print("=========================================================================")

# Watchlist Conversion

In [0]:
# How to calculate: Watched / Watchlist entries * 100
# Why it matters: Target > 40%. Intent turning into viewing.

# 1. Total watchlist entries and converted (watched) entries
total_entries = len(watchlist_df)
watched_entries = watchlist_df['watched'].sum()  # counts True values

# 2. Calculate Watchlist Conversion Rate (%)
conversion_rate = (watched_entries / total_entries) * 100

# 3. Evaluate against target benchmark (> 40%)
target_met = conversion_rate > 40.0
status = "Healthy (Above Target)" if target_met else "At Risk (Below Target)"

# Output results
print("================ WATCHLIST CONVERSION METRICS ================")
print(f"Total Watchlist Entries : {total_entries:,}")
print(f"Watched (Converted)     : {watched_entries:,}")
print(f"Unwatched Watchlist Items: {total_entries - watched_entries:,}")
print(f"Conversion Rate         : {conversion_rate:.2f}%")
print(f"Target Benchmark        : > 40.0%")
print(f"Health Status           : {status}")
print("==============================================================")

# Hit concentration

In [0]:

# Plays from top 10% titles / total * 100
# Concetration risk, if the few titles carries all the plays.

# 1. Determine the top 10% threshold of titles
catalog_size = len(titles_df)
top_10_count = int(catalog_size * 0.10)  # 10% of catalog

# 2. Extract the top 10% titles by total_plays
top_10_titles = titles_df.nlargest(top_10_count, 'total_plays')

# 3. Sum plays for the top 10% and the entire catalog
top_10_plays = top_10_titles['total_plays'].sum()
total_plays = titles_df['total_plays'].sum()

# 4. Calculate Hit Concentration (%)
hit_concentration_pct = (top_10_plays / total_plays) * 100

# Output results
print("================ HIT CONCENTRATION (TITLES_DF) ================")
print(f"Total Catalog Titles     : {catalog_size:,}")
print(f"Top 10% Title Cutoff     : Top {top_10_count:,} titles")
print(f"Plays from Top 10% Titles: {top_10_plays:,}")
print(f"Total Platform Plays     : {total_plays:,}")
print(f"Hit Concentration        : {hit_concentration_pct:.2f}%")
print("===============================================================")

# Original Share of Hours

In [0]:

# How to calcuate: Hours on Originals / total hours * 100
# Why it matters: How much engagement originals drives

# 1. Filter watch hours where license_type is 'Original' (case-insensitive check)
is_original_license = titles_df['license_type'].str.strip().str.lower() == 'original'
original_hours = titles_df.loc[is_original_license, 'total_watch_hours'].sum()

# 2. Total watch hours across all catalog titles
total_hours = titles_df['total_watch_hours'].sum()

# 3. Calculate Original Share of Hours (%)
original_share_hours = (original_hours / total_hours) * 100

# Output results
print("================ ORIGINAL SHARE OF HOURS ================")
print(f"Original Titles Count       : {is_original_license.sum():,}")
print(f"Original Watch Hours        : {original_hours:,.2f} hours")
print(f"Total Platform Watch Hours  : {total_hours:,.2f} hours")
print(f"Original Share of Hours     : {original_share_hours:.2f}%")
print("=========================================================")